In [0]:
import requests
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql import DataFrame
from functools import reduce
from delta.tables import DeltaTable
import requests, zipfile, io
import re
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from pathlib import Path
from datetime import datetime
import pprint
import os
import json
import time

## IBOV

### 1. Verificando os arquivos salvando em uma camada intermediaria

In [0]:
url = "https://query1.finance.yahoo.com/v8/finance/chart/%5EBVSP?range=10y&interval=1d"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)

data = response.json()

timestamps = data["chart"]["result"][0]["timestamp"]
close = data["chart"]["result"][0]["indicators"]["quote"][0]["close"]

# criar lista de registros
records = list(zip(timestamps, close))

# criar dataframe spark
df_ibov = spark.createDataFrame(records, ["timestamp", "close"])

file_path_ibov = "/Volumes/workspace/case_spark_cvm/raw/data_ibov/"
df_ibov.write.mode('overwrite').parquet(file_path_ibov)

### 2. Salvar em camada Bronze Particionada

In [0]:
df_ibov_bronze = spark.read.parquet("/Volumes/workspace/case_spark_cvm/raw/data_ibov/")

df_ibov_bronze = df_ibov_bronze.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_ibov_bronze.write \
    .mode('overwrite') \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .partitionBy('data_processamento')\
    .format('delta')\
    .save("/Volumes/workspace/case_spark_cvm/bronze/data_ibov/")

In [0]:
display(df_ibov_bronze)